# Training a larger transformer on enwik8

Longer context (512 vs. 256), deeper model (6 vs. 4 layers), full 55MB of enwik8 (50MB train / 5MB held out) instead of 5MB.

**GPU**: `Runtime > Change runtime type > T4 GPU`

Checkpoints every 5000 steps.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('Warning: GPU is not enabled!')
else:
    print('GPU:', torch.cuda.get_device_name(0))
torch.manual_seed(11)

device: cuda
GPU: Tesla T4


## Data
Upload `enwik8_first_55MB.txt`.

In [2]:
from google.colab import files
uploaded = files.upload()

Saving enwik8_first_55MB.txt to enwik8_first_55MB.txt


In [3]:
with open('enwik8_first_55MB.txt', 'rb') as f:
    raw = f.read()
print('total bytes:', len(raw))

data = np.frombuffer(raw, dtype=np.uint8).astype(np.int64)

TRAIN_BYTES = 50_000_000
HELDOUT_BYTES = 5_000_000
train_data = torch.tensor(data[:TRAIN_BYTES], dtype=torch.long)
heldout_data = torch.tensor(data[TRAIN_BYTES:TRAIN_BYTES + HELDOUT_BYTES], dtype=torch.long)
print('train:', len(train_data), ' heldout (completely fresh):', len(heldout_data))

total bytes: 57671680
train: 50000000  heldout (completely fresh): 5000000


## Architecture
6 layers, 6 heads, embedding dim 192, context 512. ~5M parameters.

In [4]:
VOCAB_SIZE = 256
BLOCK_SIZE = 512
N_LAYER = 6
N_HEAD = 6
N_EMBD = 192
DROPOUT = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(N_EMBD, 3*N_EMBD)
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.n_head = N_HEAD
        self.drop = nn.Dropout(DROPOUT)
        mask = torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)).view(1, 1, BLOCK_SIZE, BLOCK_SIZE)
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(N_EMBD, dim=2)
        hd = C // self.n_head
        q = q.view(B, T, self.n_head, hd).transpose(1, 2)
        k = k.view(B, T, self.n_head, hd).transpose(1, 2)
        v = v.view(B, T, self.n_head, hd).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / (hd ** 0.5)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.drop(att)
        out = att @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out)

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ln2 = nn.LayerNorm(N_EMBD)
        self.mlp = nn.Sequential(
            nn.Linear(N_EMBD, 4*N_EMBD), nn.GELU(),
            nn.Linear(4*N_EMBD, N_EMBD), nn.Dropout(DROPOUT)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok_emb = nn.Embedding(VOCAB_SIZE, N_EMBD)
        self.pos_emb = nn.Parameter(torch.zeros(1, BLOCK_SIZE, N_EMBD))
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.tok_emb(idx) + self.pos_emb[:, :T, :]
        x = self.drop(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), targets.view(-1), reduction='mean')
        return logits, loss

model = TinyGPT().to(device)
n_params = sum(p.numel() for p in model.parameters())
print('number of parameters:', n_params)

number of parameters: 2866176


## Training
`MAX_STEPS = 100` for a quick check, 25,000 for a real run. Can be interrupted anytime - the loss-extraction cell only needs `model`.

Checkpoints download every 5000 steps.

In [5]:
def get_batch(data_split, batch_size):
    ix = torch.randint(0, len(data_split) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([data_split[i:i+BLOCK_SIZE] for i in ix])
    y = torch.stack([data_split[i+1:i+BLOCK_SIZE+1] for i in ix])
    return x.to(device), y.to(device)

BATCH_SIZE = 48
MAX_STEPS = 25000
LR = 3e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
t0 = time.time()
model.train()
for step in range(MAX_STEPS):
    xb, yb = get_batch(train_data, BATCH_SIZE)
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 200 == 0 or step == MAX_STEPS - 1:
        elapsed = time.time() - t0
        eta = elapsed / (step+1) * (MAX_STEPS - step - 1)
        print(f'step {step:6d}  loss {loss.item():.4f}  bits/byte {loss.item()/0.6931:.4f}  '
              f'elapsed {elapsed:.0f}s  eta {eta:.0f}s')
    if step > 0 and step % 5000 == 0:
        torch.save(model.state_dict(), f'checkpoint_step{step}.pt')
        from google.colab import files as _files
        _files.download(f'checkpoint_step{step}.pt')
        print('checkpoint saved and downloaded: step', step)

print('training finished. total time:', round(time.time()-t0), 'seconds')

step      0  loss 5.6641  bits/byte 8.1721  elapsed 1s  eta 25853s
step    200  loss 2.6292  bits/byte 3.7933  elapsed 82s  eta 10082s
step    400  loss 2.5459  bits/byte 3.6732  elapsed 175s  eta 10740s
step    600  loss 2.5447  bits/byte 3.6715  elapsed 272s  eta 11024s
step    800  loss 2.4626  bits/byte 3.5530  elapsed 369s  eta 11134s
step   1000  loss 2.2724  bits/byte 3.2786  elapsed 465s  eta 11152s
step   1200  loss 2.1540  bits/byte 3.1077  elapsed 562s  eta 11140s
step   1400  loss 1.9852  bits/byte 2.8642  elapsed 658s  eta 11092s
step   1600  loss 1.9365  bits/byte 2.7939  elapsed 755s  eta 11035s
step   1800  loss 1.9476  bits/byte 2.8100  elapsed 851s  eta 10967s
step   2000  loss 1.8452  bits/byte 2.6623  elapsed 948s  eta 10891s
step   2200  loss 1.8212  bits/byte 2.6276  elapsed 1044s  eta 10812s
step   2400  loss 1.8010  bits/byte 2.5985  elapsed 1140s  eta 10730s
step   2600  loss 1.7221  bits/byte 2.4847  elapsed 1236s  eta 10646s
step   2800  loss 1.6290  bits/byt

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

checkpoint saved and downloaded: step 5000
step   5200  loss 1.4628  bits/byte 2.1106  elapsed 2487s  eta 9468s
step   5400  loss 1.4232  bits/byte 2.0534  elapsed 2583s  eta 9373s
step   5600  loss 1.3520  bits/byte 1.9507  elapsed 2679s  eta 9278s
step   5800  loss 1.4106  bits/byte 2.0352  elapsed 2775s  eta 9185s
step   6000  loss 1.3822  bits/byte 1.9943  elapsed 2872s  eta 9093s
step   6200  loss 1.3927  bits/byte 2.0094  elapsed 2969s  eta 9001s
step   6400  loss 1.2825  bits/byte 1.8504  elapsed 3066s  eta 8908s
step   6600  loss 1.3306  bits/byte 1.9197  elapsed 3162s  eta 8814s
step   6800  loss 1.2416  bits/byte 1.7914  elapsed 3259s  eta 8720s
step   7000  loss 1.3466  bits/byte 1.9428  elapsed 3355s  eta 8625s
step   7200  loss 1.3628  bits/byte 1.9663  elapsed 3451s  eta 8529s
step   7400  loss 1.2456  bits/byte 1.7971  elapsed 3547s  eta 8435s
step   7600  loss 1.2635  bits/byte 1.8229  elapsed 3644s  eta 8342s
step   7800  loss 1.2704  bits/byte 1.8330  elapsed 3741s  e

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

checkpoint saved and downloaded: step 15000
step  15200  loss 1.1945  bits/byte 1.7234  elapsed 7325s  eta 4722s
step  15400  loss 1.1711  bits/byte 1.6897  elapsed 7421s  eta 4626s
step  15600  loss 1.2685  bits/byte 1.8301  elapsed 7519s  eta 4530s
step  15800  loss 1.2619  bits/byte 1.8206  elapsed 7615s  eta 4434s
step  16000  loss 1.1785  bits/byte 1.7003  elapsed 7712s  eta 4337s
step  16200  loss 1.1896  bits/byte 1.7164  elapsed 7808s  eta 4241s
step  16400  loss 1.2502  bits/byte 1.8038  elapsed 7905s  eta 4144s
step  16600  loss 1.1841  bits/byte 1.7084  elapsed 8001s  eta 4048s
step  16800  loss 1.2166  bits/byte 1.7553  elapsed 8099s  eta 3952s
step  17000  loss 1.2097  bits/byte 1.7453  elapsed 8196s  eta 3856s
step  17200  loss 1.1649  bits/byte 1.6808  elapsed 8292s  eta 3760s
step  17400  loss 1.1772  bits/byte 1.6984  elapsed 8388s  eta 3663s
step  17600  loss 1.1864  bits/byte 1.7117  elapsed 8485s  eta 3567s
step  17800  loss 1.1375  bits/byte 1.6412  elapsed 8582s  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

checkpoint saved and downloaded: step 20000
step  20200  loss 1.2457  bits/byte 1.7972  elapsed 9745s  eta 2315s
step  20400  loss 1.1317  bits/byte 1.6328  elapsed 9842s  eta 2219s
step  20600  loss 1.2036  bits/byte 1.7365  elapsed 9938s  eta 2122s
step  20800  loss 1.0976  bits/byte 1.5836  elapsed 10035s  eta 2026s
step  21000  loss 1.1281  bits/byte 1.6276  elapsed 10131s  eta 1929s
step  21200  loss 1.1450  bits/byte 1.6520  elapsed 10228s  eta 1833s
step  21400  loss 1.1084  bits/byte 1.5992  elapsed 10325s  eta 1736s
step  21600  loss 1.2243  bits/byte 1.7665  elapsed 10422s  eta 1640s
step  21800  loss 1.2746  bits/byte 1.8390  elapsed 10519s  eta 1543s
step  22000  loss 1.2241  bits/byte 1.7661  elapsed 10615s  eta 1447s
step  22200  loss 1.1753  bits/byte 1.6957  elapsed 10712s  eta 1351s
step  22400  loss 1.1963  bits/byte 1.7260  elapsed 10809s  eta 1254s
step  22600  loss 1.1728  bits/byte 1.6922  elapsed 10906s  eta 1158s
step  22800  loss 1.1317  bits/byte 1.6328  elaps

### Resuming from a checkpoint
```python
from google.colab import files
uploaded = files.upload()  # checkpoint_stepXXXX.pt
model.load_state_dict(torch.load('checkpoint_stepXXXX.pt'))
```

## Extracting per-byte loss
Same as before, context 512, on 2MB of fresh held-out data.

In [6]:
model.eval()
n_chunks = len(heldout_data) // BLOCK_SIZE
per_position_bits = np.zeros(n_chunks * (BLOCK_SIZE - 1), dtype=np.float32)

with torch.no_grad():
    for c in range(n_chunks):
        chunk = heldout_data[c*BLOCK_SIZE:(c+1)*BLOCK_SIZE]
        x = chunk[:-1].unsqueeze(0).to(device)
        y = chunk[1:].unsqueeze(0).to(device)
        logits, _ = model(x)
        logp = F.log_softmax(logits, dim=-1)
        token_logp = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1).squeeze(0)
        bits = (-token_logp / np.log(2)).cpu().numpy()

        start = c*(BLOCK_SIZE-1)
        per_position_bits[start:start+(BLOCK_SIZE-1)] = bits
        if c % 200 == 0:
            print('chunk', c, '/', n_chunks)

print('mean bits per byte:', per_position_bits.mean())

chunk 0 / 9765
chunk 200 / 9765
chunk 400 / 9765
chunk 600 / 9765
chunk 800 / 9765
chunk 1000 / 9765
chunk 1200 / 9765
chunk 1400 / 9765
chunk 1600 / 9765
chunk 1800 / 9765
chunk 2000 / 9765
chunk 2200 / 9765
chunk 2400 / 9765
chunk 2600 / 9765
chunk 2800 / 9765
chunk 3000 / 9765
chunk 3200 / 9765
chunk 3400 / 9765
chunk 3600 / 9765
chunk 3800 / 9765
chunk 4000 / 9765
chunk 4200 / 9765
chunk 4400 / 9765
chunk 4600 / 9765
chunk 4800 / 9765
chunk 5000 / 9765
chunk 5200 / 9765
chunk 5400 / 9765
chunk 5600 / 9765
chunk 5800 / 9765
chunk 6000 / 9765
chunk 6200 / 9765
chunk 6400 / 9765
chunk 6600 / 9765
chunk 6800 / 9765
chunk 7000 / 9765
chunk 7200 / 9765
chunk 7400 / 9765
chunk 7600 / 9765
chunk 7800 / 9765
chunk 8000 / 9765
chunk 8200 / 9765
chunk 8400 / 9765
chunk 8600 / 9765
chunk 8800 / 9765
chunk 9000 / 9765
chunk 9200 / 9765
chunk 9400 / 9765
chunk 9600 / 9765
mean bits per byte: 1.6832635


In [7]:
np.save('neural_lm_losses_v2.npy', per_position_bits)
np.save('neural_lm_meta_v2.npy', np.array([BLOCK_SIZE, n_chunks, N_LAYER, N_HEAD, N_EMBD, n_params]))

from google.colab import files
files.download('neural_lm_losses_v2.npy')
files.download('neural_lm_meta_v2.npy')
print('ready - please send me these two files')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

ready - please send me these two files


## Extra validation: shuffled control + matched-range comparison

Same two checks as `train_neural_lm.ipynb`, run here with this (large) model.

In [8]:
def spectrum_zscore(cost_signal, target_period):
    sig = cost_signal - cost_signal.mean()
    spec = np.abs(np.fft.rfft(sig))
    freqs = np.fft.rfftfreq(len(sig), d=1.0)
    idx = np.argmin(np.abs(freqs - 1.0/target_period))
    others = np.delete(spec[1:], idx-1)
    z = (spec[idx]-others.mean())/others.std()
    return z

def extract_losses(model, heldout_tensor):
    model.eval()
    n_chunks = len(heldout_tensor) // BLOCK_SIZE
    bits = np.zeros(n_chunks * (BLOCK_SIZE - 1), dtype=np.float32)
    with torch.no_grad():
        for c in range(n_chunks):
            chunk = heldout_tensor[c*BLOCK_SIZE:(c+1)*BLOCK_SIZE]
            x = chunk[:-1].unsqueeze(0).to(device)
            y = chunk[1:].unsqueeze(0).to(device)
            logits, _ = model(x)
            logp = F.log_softmax(logits, dim=-1)
            token_logp = logp.gather(-1, y.unsqueeze(-1)).squeeze(-1).squeeze(0)
            bits[c*(BLOCK_SIZE-1):(c+1)*(BLOCK_SIZE-1)] = (-token_logp / np.log(2)).cpu().numpy()
    return bits

# --- shuffled control ---
real_losses = per_position_bits  # already computed above
z_real = spectrum_zscore(real_losses, BLOCK_SIZE - 1)

heldout_shuf = heldout_data[torch.randperm(len(heldout_data))]
shuf_losses = extract_losses(model, heldout_shuf)
z_shuf = spectrum_zscore(shuf_losses, BLOCK_SIZE - 1)

print(f'real held-out:     mean={real_losses.mean():.4f} bits/byte, context-reset z={z_real:.2f}')
print(f'shuffled control:  mean={shuf_losses.mean():.4f} bits/byte, context-reset z={z_shuf:.2f}')

real held-out:     mean=1.6833 bits/byte, context-reset z=104.53
shuffled control:  mean=8.3061 bits/byte, context-reset z=96.45


In [9]:
# --- matched-range comparison vs. the small model ---
MATCH_START, MATCH_END = 55_000_000, 57_671_680
matched_bytes = torch.tensor(data[MATCH_START:MATCH_END], dtype=torch.long)
matched_losses = extract_losses(model, matched_bytes)
print(f'large model on the matched range [{MATCH_START:,}:{MATCH_END:,}]: {matched_losses.mean():.4f} bits/byte')

np.save('large_model_matched_losses.npy', matched_losses)
from google.colab import files
files.download('large_model_matched_losses.npy')
print('saved -- compare this mean directly to the one from train_neural_lm.ipynb\'s matching cell')

large model on the matched range [55,000,000:57,671,680]: 1.6468 bits/byte


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

saved -- compare this mean directly to the one from train_neural_lm.ipynb's matching cell
